# 02 · Signal Cleaning, QC & Modeling Hand-off
**BCI Motor Imagery (Left / Right) · Cellula EEG · 4 channels (FZ, C3, CZ, C4)**

This notebook **continues `01_Data_Collection_and_Validation`**. Notebook 01 validated the files and documented the
problems. This notebook **fixes them** and produces the clean arrays the modeling team needs.

| # | Section | Output |
|---|---|---|
| 0 | Setup & configuration | one `CFG` dict, one set of paths |
| 1 | Load & unified index | `meta` table keyed by `trial_id` (1-based, = file number) |
| 2 | Sampling-rate evidence | fs = 250 Hz justified from the data |
| 2b | Recording continuity | files are consecutive 10 s slices → recordings & filter blocks from the `Time` column |
| 3 | Stale-sample (plateau) analysis | `stale_report.csv` |
| 4 | Cleaning pipeline | interpolation → **continuous-stream** notch + band-pass → rejection |
| 5 | Before / after QC | figures |
| 6 | Sessions & gain regimes | `session_id` (recording) for leakage-free CV, `regime_id` for normalization |
| 7 | Neurophysiology checks | PSD per class, lateralization, compact features |
| 8 | Decodability check | CSP + LDA with shifted / permuted label controls |
| 8b | Where could the signal hide? | within-session features, time-resolved AUC, windowed CSP |
| 9 | Hand-off package | `X_clean.npy`, `y.npy`, `trials_*.csv`, `manifest.json` |
| 10 | Decision log & open questions | what is settled and what is not |

**How to run:** set `PROJECT_DIR` in section 0, then *Run All*. Everything is reproducible (`CFG["seed"]`).
All outputs go to `outputs_clean/`, so nothing from notebook 01 is overwritten.

## 0 · Setup & configuration

In [ ]:
import os, re, json, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, iirnotch, sosfiltfilt, filtfilt, welch, find_peaks
from scipy.integrate import trapezoid
from scipy.ndimage import binary_dilation
from scipy.stats import mannwhitneyu

try:
    from tqdm.auto import tqdm
except ImportError:                        # tqdm is optional
    def tqdm(x, **kwargs):
        return x
try:
    display
except NameError:                          # running outside Jupyter
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------------------------------------------------------------- paths (edit PROJECT_DIR only)
PROJECT_DIR = Path(os.environ.get("BCI_PROJECT_DIR", r"C:\Users\noura\Z-BCI-Motor"))
DATA_DIR    = PROJECT_DIR / "ML CSV"
LABELS_PATH = PROJECT_DIR / "labels.csv"
OUT_DIR     = PROJECT_DIR / "outputs_clean"
FIG_DIR     = OUT_DIR / "figures"
for d in (OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

CHANNELS = ["FZ", "C3", "CZ", "C4"]
N_SAMPLES = 2500

# ---------------------------------------------------------------- one place for every parameter
CFG = dict(
    fs=250.0,                       # justified in section 2
    notch_hz=50.0, notch_q=30.0,    # mains frequency (Egypt = 50 Hz)
    band=(1.0, 40.0), band_order=4, # band-pass; modeling can narrow to 8-30 Hz for CSP
    max_gap_s=0.5,                  # Time gap between consecutive files above this = new recording (section 2b)
    max_dc_step=15.0,               # DC jump between consecutive files (in local-amplitude units) = new filter block
    # --- rejection rules (tune after looking at section 4 / 5 plots)
    max_stale_frac=0.15,            # > 15 % stale samples  -> reject
    max_stale_run=50,               # one stale run > 50 samples (0.2 s) -> reject
    contam_pad_s=0.2,               # stale runs also contaminate +-0.2 s around them (filter ringing)
    max_contam_frac=0.15,           # > 15 % of the trial contaminated -> reject
    max_abs_z=10.0,                 # transient spike inside a trial (gain-invariant) -> reject
    min_std=1e-6,                   # flat channel -> reject
    # --- spectral analysis
    psd_nperseg_s=2.0, n_psd_check=200,
    bands={"delta": (1, 4), "theta": (4, 8), "mu": (8, 12), "beta_low": (13, 20),
           "beta_high": (20, 30), "beta": (13, 30), "gamma": (30, 40)},
    # --- diagnostics
    csp_band=(8.0, 30.0), csp_pairs=2, cv_folds=5, n_perm=10,
    session_win=5, session_z=6.0, session_min_gap=20,
    seed=0,
)
RNG = np.random.default_rng(CFG["seed"])
print("PROJECT_DIR:", PROJECT_DIR)
print("Outputs     :", OUT_DIR)

## 1 · Load & unified index

**One index convention for the whole project:** `trial_id` = the number in the file name (**1-based**).
Notebook 01 mixed 0-based (quality reports) and 1-based (features / labels) indices, which silently shifts quality
flags by one trial when tables are merged. From here on, **always merge on `trial_id` or `file_name`**, never on the
row position.

> ✅ **Label mapping confirmed by the project owner:** label *i* ↔ file *i* (the first 2160 of the 2400 labels).
> Section 8 therefore acts as a *decodability* check (is there class information at all?), not as an alignment check.

In [ ]:
def file_number(p):
    return int(re.search(r"(\d+)$", Path(p).stem).group(1))

files = sorted(DATA_DIR.glob("cellula_MI_data_*.csv"), key=file_number)
assert files, f"No CSV files found in {DATA_DIR}"
names = np.array([f.name for f in files])

labels_all = pd.read_csv(LABELS_PATH)["label"].astype(str).str.strip()
print(f"EEG files: {len(files)} | labels: {len(labels_all)} "
      f"({labels_all.value_counts().to_dict()})")

# ---- read all CSVs once, cache as .npz (cache is invalidated if the file list changes)
cache = OUT_DIR / "raw_cache.npz"
X_raw = T_raw = None
if cache.exists():
    z = np.load(cache, allow_pickle=False)
    if np.array_equal(z["names"], names):
        X_raw, T_raw = z["X"], z["T"]
        print("Loaded raw cache.")
if X_raw is None:
    X_raw = np.empty((len(files), N_SAMPLES, 4), dtype=np.float32)
    T_raw = np.empty((len(files), N_SAMPLES), dtype=np.float64)
    for i, f in enumerate(tqdm(files, desc="Reading CSVs")):
        df = pd.read_csv(f)
        assert list(df.columns) == ["Time"] + CHANNELS, f"unexpected columns in {f.name}"
        assert len(df) == N_SAMPLES, f"unexpected length in {f.name}: {len(df)}"
        X_raw[i] = df[CHANNELS].to_numpy(np.float32)
        T_raw[i] = df["Time"].to_numpy(np.float64)
    np.savez(cache, X=X_raw, T=T_raw, names=names)
    print("Read CSVs and wrote cache.")

# ---- the single metadata table used everywhere below
meta = pd.DataFrame({"trial_id": [file_number(f) for f in files], "file_name": names})
assert meta.trial_id.max() <= len(labels_all), "more files than labels"
meta["label"] = [labels_all.iloc[n - 1] for n in meta.trial_id]          # label i <-> file i (confirmed by the owner)
meta["y"] = (meta.label.str.lower() == "right").astype(int)              # Left=0, Right=1

print("X_raw:", X_raw.shape, X_raw.dtype)
print("trial_id contiguous 1..N:", np.array_equal(meta.trial_id.values, np.arange(1, len(meta) + 1)))
print("Class balance:", meta.label.value_counts().to_dict())
display(meta.head())

## 2 · Sampling-rate evidence

Notebook 01 left the sampling rate as *unconfirmed*. Two independent pieces of evidence settle it:

1. **Duration-based rate:** 2500 samples span ≈ 10 s of the `Time` column → ≈ 250 Hz.
2. **Mains peak:** if samples are index-uniform at 250 Hz, power-line noise must show up at **50 Hz** (Egypt's mains).
   A wrong `fs` would move the peak (e.g. 256 Hz would put it at 48.8 Hz).

Individual `Time` intervals are irregular because `Time` records *when the buffer was read*, not when the sample was
taken. The signal itself is periodic in **index** space, so we **treat the sample index as time** and do **not**
resample on the timestamps.

In [ ]:
dur = T_raw[:, -1] - T_raw[:, 0]
eff_rate = (N_SAMPLES - 1) / dur
print("Duration-based rate (Hz):  median = %.2f | IQR = [%.2f, %.2f] | min = %.1f | max = %.1f"
      % (np.median(eff_rate), *np.percentile(eff_rate, [25, 75]), eff_rate.min(), eff_rate.max()))

# dominant frequency per channel on a random subset (index spacing at fs = 250)
sel = RNG.choice(len(X_raw), size=min(CFG["n_psd_check"], len(X_raw)), replace=False)
peaks = []
for i in sel:
    x = X_raw[i].astype(np.float64)
    x -= x.mean(0)
    f, P = welch(x, fs=CFG["fs"], nperseg=1024, noverlap=512, axis=0)
    peaks.extend(f[P.argmax(axis=0)])
peaks = np.array(peaks)
share_50 = np.mean((peaks > 49.5) & (peaks < 50.6))
print(f"Channel-trials whose dominant peak is within 49.5-50.6 Hz: {share_50:.1%} (n={len(peaks)})")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(eff_rate, bins=60, color="C0"); ax[0].axvline(250, color="k", ls="--")
ax[0].set(title="Duration-based sampling rate per trial", xlabel="Hz")
ax[1].hist(peaks, bins=np.arange(0, 125, 0.5), color="C1"); ax[1].axvline(50, color="k", ls="--")
ax[1].set(title="Dominant spectral peak (fs = 250 Hz assumed)", xlabel="Hz", yscale="log")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_fs_evidence.png"); plt.show()
CFG["fs_evidence"] = {"median_duration_rate_hz": float(np.median(eff_rate)),
                      "share_peak_at_50hz": float(share_50)}

## 2b · Recording continuity (key finding)

Consecutive files are **back-to-back 10 s slices of a continuous recording**: the last `Time` value of file *n* equals the first
of file *n+1* (gap ≈ 0). Consequences:

* **Filter the continuous stream, not each file.** Filtering every 10 s slice separately creates edge transients
  (the 1–2 s decaying "start transient" seen in notebook 02 v1) and wastes data. Here every continuous *block* is filtered as one signal.
* **Exact recordings from the clock:** a big jump in `Time` (or a break in the file numbering) means a **new recording session**.
  This is a far better `session_id` for cross-validation than any amplitude heuristic.
* **Filter blocks:** a sudden DC step between two consecutive files would make the filters ring, so the stream is also cut there.
  The first and last trial of every block are rejected (they carry the filter's edge effect).

In [ ]:
ids = meta.trial_id.to_numpy()
gap = T_raw[1:, 0] - T_raw[:-1, -1]                    # s between end of file i and start of file i+1

K = 50                                                  # samples used to estimate the level at each file edge
first = np.median(X_raw[1:, :K, :], axis=1).astype(np.float64)
last = np.median(X_raw[:-1, -K:, :], axis=1).astype(np.float64)
amp_raw = np.median(np.abs(X_raw - np.median(X_raw, axis=1, keepdims=True)), axis=1).astype(np.float64)
step = (np.abs(first - last) / (np.maximum(amp_raw[1:], amp_raw[:-1]) + 1e-9)).max(axis=1)   # DC jump in amplitude units

gap_break = np.abs(gap) > CFG["max_gap_s"]
id_break = np.diff(ids) != 1
step_break = step > CFG["max_dc_step"]
rec_break = np.r_[True, gap_break | id_break]            # new RECORDING (time gap / missing file)
blk_break = rec_break | np.r_[False, step_break]         # new FILTER BLOCK (also DC steps)
rec_id = np.cumsum(rec_break) - 1
block_id = np.cumsum(blk_break) - 1

print(f"Median |gap| between consecutive files: {np.median(np.abs(gap)):.4f} s | "
      f"{np.mean(np.abs(gap) < 0.05):.1%} of pairs have |gap| < 50 ms")
print(f"Breaks -> time gap: {int(gap_break.sum())} | numbering gap: {int(id_break.sum())} | DC step: {int(step_break.sum())}")
print(f"Recordings: {rec_id.max() + 1} | filter blocks: {block_id.max() + 1}")
rec_tbl = (pd.DataFrame({"recording_id": rec_id, "trial_id": ids})
           .groupby("recording_id").agg(first_trial=("trial_id", "min"), last_trial=("trial_id", "max"), n_trials=("trial_id", "size")))
rec_tbl["start_time_s"] = [T_raw[np.flatnonzero(rec_id == r)[0], 0] for r in rec_tbl.index]
display(rec_tbl.head(30))
bs = pd.Series(block_id).value_counts()
print(f"Block sizes (trials): median {bs.median():.0f}, min {bs.min()}, max {bs.max()}; "
      f"trials that sit on a block edge (will be rejected): {int(2 * len(bs) - (bs == 1).sum())} (upper bound)")

fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(ids[1:], np.abs(gap) + 1e-6, lw=.6); ax[0].axhline(CFG["max_gap_s"], color="r", ls="--", lw=.8)
ax[0].set(yscale="log", title="|Time gap| between consecutive files (s)")
ax[1].plot(ids[1:], step + 1e-3, lw=.6, color="C1"); ax[1].axhline(CFG["max_dc_step"], color="r", ls="--", lw=.8)
ax[1].set(yscale="log", title="DC step between consecutive files (local-amplitude units)", xlabel="trial_id")
plt.tight_layout(); plt.savefig(FIG_DIR / "02b_continuity.png"); plt.show()

## 3 · Stale-sample (plateau) analysis

A **stale row** is a row whose four channel values are *identical* to the previous row. In the raw data these form long
plateaus whose timestamps span only ≈ 0.0001 s for hundreds of samples, i.e. the device re-sent the last value (buffer
hold / export artifact). They are **not** EEG. Strategy: interpolate the runs so the filters can run, then **reject trials
that are too contaminated**. Interpolating across a hold creates a step in the 50 Hz oscillation, which rings through the
filters, so the area **±0.2 s around each stale run** is counted as contaminated too (`contam_frac`).

In [ ]:
def stale_mask(x):
    """True where all channels equal the previous row (hold / stale buffer)."""
    m = np.zeros(len(x), dtype=bool)
    m[1:] = (np.diff(x, axis=0) == 0).all(axis=1)
    return m

def run_lengths(mask):
    if not mask.any():
        return np.array([], dtype=int)
    d = np.diff(np.r_[0, mask.astype(int), 0])
    return np.flatnonzero(d == -1) - np.flatnonzero(d == 1)

PAD = int(CFG["contam_pad_s"] * CFG["fs"])
rows = []
for i in range(len(X_raw)):
    m = stale_mask(X_raw[i])
    r = run_lengths(m)
    contam = binary_dilation(m, structure=np.ones(2 * PAD + 1, bool)).mean() if m.any() else 0.0
    rows.append((r.sum() / N_SAMPLES, int(r.max()) if len(r) else 0, len(r), contam))
stale_df = pd.DataFrame(rows, columns=["stale_frac", "max_stale_run", "n_stale_runs", "contam_frac"])
stale_df.insert(0, "trial_id", meta.trial_id.values)
stale_df.insert(1, "file_name", meta.file_name.values)
stale_df.to_csv(OUT_DIR / "stale_report.csv", index=False)

summary = pd.Series({
    "trials with any stale rows": int((stale_df.stale_frac > 0).sum()),
    "stale_frac > 5 %":  int((stale_df.stale_frac > .05).sum()),
    "stale_frac > 15 % (reject rule)": int((stale_df.stale_frac > CFG["max_stale_frac"]).sum()),
    "max run > 50 samples (reject rule)": int((stale_df.max_stale_run > CFG["max_stale_run"]).sum()),
    "max run > 100 samples": int((stale_df.max_stale_run > 100).sum()),
}, name="trials")
display(summary.to_frame())

# how many trials would each contamination threshold reject? (stale rows + their +-0.2 s surroundings)
thr_tbl = pd.DataFrame({"contam_frac >": [.05, .10, .15, .20, .30]})
thr_tbl["trials rejected"] = [int((stale_df.contam_frac > t).sum()) for t in thr_tbl["contam_frac >"]]
thr_tbl["% of all trials"] = (thr_tbl["trials rejected"] / len(stale_df) * 100).round(1)
display(thr_tbl)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].hist(stale_df.stale_frac * 100, bins=50); ax[0].set(title="Stale samples per trial (%)", yscale="log")
ax[1].hist(stale_df.max_stale_run, bins=50); ax[1].set(title="Longest stale run (samples)", yscale="log")
ax[2].plot(stale_df.trial_id, stale_df.stale_frac * 100, lw=.6)
ax[2].set(title="Stale % along the recording order", xlabel="trial_id")
plt.tight_layout(); plt.savefig(FIG_DIR / "03_stale_samples.png"); plt.show()

## 4 · Cleaning pipeline

All parameters come from `CFG`. Because the files are continuous (section 2b), the filters run on **whole blocks**:

1. **Interpolate** stale runs linearly (per trial and channel) so the signal is defined everywhere.
2. **Filter each continuous block as one stream:** subtract one constant, **notch** 50 Hz (`iirnotch`, Q = 30), **band-pass** 1–40 Hz
   (Butterworth order 4), both zero-phase. No trimming inside a block, so every trial keeps its full **10 s (2500 samples)**.
3. **Reject** trials that: sit at the **edge of a block**, have too many / too long stale runs, have a large contaminated area
   (±0.2 s around stale runs), contain a flat channel, or contain a transient spike.

Then two versions are produced: `X_clean` (no re-reference) and `X_clean_car` (common average reference).
Each trial is divided by **one robust scalar** (square root of its *median* power over time, so bursts do not distort it), so **inter-channel variance ratios are preserved**
(per-channel z-scoring would destroy what CSP uses).

In [ ]:
NOTCH = iirnotch(CFG["notch_hz"], CFG["notch_q"], fs=CFG["fs"])
SOS = butter(CFG["band_order"], CFG["band"], btype="bandpass", fs=CFG["fs"], output="sos")

def interpolate_stale(x, mask):
    if not mask.any():
        return x
    idx = np.arange(len(x)); good = ~mask
    out = x.copy()
    for c in range(x.shape[1]):
        out[:, c] = np.interp(idx, idx[good], x[good, c])
    return out

# ---- 1) stale interpolation per trial
Xi = np.empty(X_raw.shape, dtype=np.float64)
for i in tqdm(range(len(X_raw)), desc="Stale interpolation"):
    x = X_raw[i].astype(np.float64)
    Xi[i] = interpolate_stale(x, stale_mask(x))

# ---- 2) continuous-stream filtering, one block at a time
Xf = np.empty_like(Xi)
for b in tqdm(np.unique(block_id), desc="Filtering blocks"):
    idx = np.flatnonzero(block_id == b)
    seg = Xi[idx].reshape(-1, 4)                       # trials of the block, concatenated in time
    seg = seg - np.median(seg, axis=0)
    seg = filtfilt(*NOTCH, seg, axis=0)
    seg = sosfiltfilt(SOS, seg, axis=0)
    Xf[idx] = seg.reshape(len(idx), N_SAMPLES, 4)
clean = Xf.astype(np.float32)                           # (n_trials, 2500, 4), full 10 s per trial

# ---- 3) per-trial QC and rejection
pos = pd.Series(block_id).groupby(block_id).cumcount().to_numpy()
size = pd.Series(block_id).groupby(block_id).transform("size").to_numpy()
is_edge = (pos == 0) | (pos == size - 1)

recs = []
for i in tqdm(range(len(Xf)), desc="QC"):
    xc = Xf[i]
    sd = xc.std(0)
    zmax = float((np.abs(xc - xc.mean(0)) / np.maximum(sd, 1e-12)).max())
    reasons = []
    if is_edge[i]:                                           reasons.append("block_edge")
    if stale_df.stale_frac.iat[i] > CFG["max_stale_frac"]:   reasons.append("stale_frac")
    if stale_df.max_stale_run.iat[i] > CFG["max_stale_run"]: reasons.append("stale_run")
    if stale_df.contam_frac.iat[i] > CFG["max_contam_frac"]: reasons.append("contaminated")
    if (sd < CFG["min_std"]).any():                          reasons.append("flat_channel")
    if zmax > CFG["max_abs_z"]:                              reasons.append("spike")
    if not np.isfinite(xc).all():                            reasons.append("non_finite")
    mad = np.median(np.abs(xc - np.median(xc, 0)), 0) * 1.4826        # robust amplitude per channel
    recs.append(dict(max_abs_z=round(zmax, 2), kept=not reasons, reject_reason="+".join(reasons),
                     clean_std_mean=float(sd.mean()), **{f"mad_{ch}": float(m) for ch, m in zip(CHANNELS, mad)}))

trials = meta.merge(stale_df.drop(columns="file_name"), on="trial_id").join(pd.DataFrame(recs))
trials["recording_id"] = rec_id
trials["block_id"] = block_id
trials["session_id"] = rec_id                           # session = recording (exact, from the Time column)
keep = trials.kept.to_numpy()
keep_idx = np.flatnonzero(keep)
Xc = clean[keep].astype(np.float64)
y = trials.loc[keep, "y"].to_numpy()

print(f"Kept {keep.sum()} / {len(trials)} trials ({keep.mean():.1%})")
reasons = trials.loc[~trials.kept, "reject_reason"].str.split("+").explode().value_counts()
display(reasons.rename("trials").to_frame())
print("Class balance after rejection:", {"Left": int((y == 0).sum()), "Right": int((y == 1).sum())})
print("max |z| percentiles (50/90/95/99):", np.round(np.percentile(trials.max_abs_z, [50, 90, 95, 99]), 1),
      "| threshold =", CFG["max_abs_z"])

## 5 · Before / after QC
Three representative trials (best session, most stale samples, noisiest raw signal) and the **median PSD** before and
after cleaning. The 50 Hz line should disappear and the 1/f shape with an alpha/mu bump should remain.

In [ ]:
std_raw = X_raw.std(axis=1).mean(1)
log_amp = np.log(trials.clean_std_mean.to_numpy())
typ = keep_idx[np.argmin(np.abs(log_amp[keep_idx] - np.median(log_amp[keep_idx])))]   # kept trial with median amplitude
examples = {"typical (median amplitude)": int(typ),
            "most stale": int(stale_df.stale_frac.values.argmax()),
            "largest raw amplitude": int(std_raw.argmax())}
off = 0                                              # cleaned trials have the same 2500 samples as raw
W = 750
fig, axes = plt.subplots(len(examples), 2, figsize=(13, 2.6 * len(examples)), sharex="col")
for r, (name, i) in enumerate(examples.items()):
    raw = X_raw[i, off:off + W] - X_raw[i].mean(0)
    cl = clean[i][:W]
    axes[r, 0].plot(raw[:, 1], lw=.7, label="C3"); axes[r, 0].plot(raw[:, 3], lw=.7, label="C4")
    axes[r, 0].set_title(f"{name} · trial_id {meta.trial_id.iat[i]} · RAW (centered)", fontsize=9)
    axes[r, 1].plot(cl[:, 1], lw=.7); axes[r, 1].plot(cl[:, 3], lw=.7)
    axes[r, 1].set_title(f"{name} · CLEANED ({'kept' if keep[i] else 'rejected: ' + trials.reject_reason.iat[i]})", fontsize=9)
axes[0, 0].legend(fontsize=8)
for a in axes[-1]: a.set_xlabel("sample")
plt.tight_layout(); plt.savefig(FIG_DIR / "05_examples_before_after.png"); plt.show()

# ---- how similar are the channels after cleaning? (common-mode / volume conduction check)
def med_corr(A, pair):
    i, j = CHANNELS.index(pair[0]), CHANNELS.index(pair[1])
    return float(np.median([np.corrcoef(a[:, i], a[:, j])[0, 1] for a in A]))
sos_8_30 = butter(4, CFG["csp_band"], btype="bandpass", fs=CFG["fs"], output="sos")
Xc_8_30 = sosfiltfilt(sos_8_30, Xc, axis=1)
Xc_8_30_car = Xc_8_30 - Xc_8_30.mean(axis=2, keepdims=True)
corr_tbl = pd.DataFrame({pair[0] + "-" + pair[1]: [med_corr(Xc, pair), med_corr(Xc_8_30, pair), med_corr(Xc_8_30_car, pair)]
                         for pair in (("C3", "C4"), ("C3", "CZ"), ("CZ", "C4"), ("FZ", "CZ"))},
                        index=["1-40 Hz", "8-30 Hz", "8-30 Hz + CAR"]).round(2)
print("Median inter-channel correlation per trial (cleaned, kept trials):")
display(corr_tbl)

# ---- median PSD before vs after (same Welch settings)
nper = int(CFG["psd_nperseg_s"] * CFG["fs"])
def psd(trial_2d):
    return welch(trial_2d, fs=CFG["fs"], nperseg=nper, noverlap=nper // 2, axis=0)

sel_k = RNG.choice(keep_idx, size=min(CFG["n_psd_check"], len(keep_idx)), replace=False)
P_raw = np.stack([psd(X_raw[i].astype(np.float64) - X_raw[i].mean(0))[1] for i in sel_k])
P_cln = np.stack([psd(clean[i].astype(np.float64))[1] for i in sel_k])
freqs = psd(clean[0].astype(np.float64))[0]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=False)
for c, ch in enumerate(CHANNELS):
    axes[c].semilogy(freqs, np.median(P_raw[:, :, c], 0), label="raw (centered)")
    axes[c].semilogy(freqs, np.median(P_cln[:, :, c], 0), label="cleaned")
    axes[c].set(title=ch, xlim=(0, 60), xlabel="Hz")
    mm = freqs <= 60
    both = np.concatenate([np.median(P_raw[:, mm, c], 0), np.median(P_cln[:, mm, c], 0)])
    axes[c].set_ylim(both.min() / 3, both.max() * 3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "05_psd_before_after.png"); plt.show()

## 6 · Sessions and gain regimes

Amplitude and DC level change in large steps along the recording order, so the data comes from **separate recordings with
different gain**. Random K-fold would put neighbours from the same recording in train *and* test and inflate accuracy.

* **`session_id` = recording** (from the `Time` clock in section 2b). Exact. Use it for `GroupKFold` and as the unit that must never be split
  across train and test.
* **`regime_id`** (below) is a **heuristic** from jumps in the per-trial DC level **and** robust (MAD) amplitude after cleaning. It captures
  *gain changes inside a recording* and is used to normalize features. **Check the plot** and adjust `CFG["session_z"]` / `CFG["session_min_gap"]`.

A *contiguous, non-shuffled* K-fold is always a safe default.

In [ ]:
def detect_sessions(dc, win, z_thr, min_gap):
    n = len(dc)
    med = np.median(dc, 0); mad = np.median(np.abs(dc - med), 0) * 1.4826 + 1e-9
    Z = (dc - med) / mad
    jump = np.zeros(n)
    for i in range(win, n - win + 1):
        jump[i] = np.abs(np.median(Z[i:i + win], 0) - np.median(Z[i - win:i], 0)).max()
    pk, _ = find_peaks(jump, height=z_thr, distance=min_gap)
    sid = np.zeros(n, dtype=int)
    for p in pk:
        sid[p:] += 1
    return sid, pk, jump

dc = X_raw.astype(np.float64).mean(axis=1)                  # (n_trials, 4) per-trial DC level
amp = np.log(trials[[f"mad_{ch}" for ch in CHANNELS]].to_numpy() + 1e-9)   # (n_trials, 4) log robust amplitude
seg_feat = np.hstack([dc, amp])                             # 8 features: DC level + gain
if len(seg_feat) >= 4 * CFG["session_win"]:
    session_all, boundaries, jump = detect_sessions(seg_feat, CFG["session_win"], CFG["session_z"], CFG["session_min_gap"])
else:
    session_all, boundaries, jump = np.zeros(len(seg_feat), dtype=int), np.array([], dtype=int), np.zeros(len(seg_feat))
trials["regime_id"] = session_all
print(f"Recordings (session_id): {trials.session_id.nunique()} | gain regimes (regime_id): {session_all.max() + 1}; regime boundaries at trial_id: "
      f"{meta.trial_id.values[boundaries].tolist()}")

fig, ax = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
for c, ch in enumerate(CHANNELS):
    ax[0].plot(meta.trial_id, dc[:, c], lw=.6, label=ch)
ax[0].set_title("Per-trial DC level (raw)"); ax[0].legend(ncol=4, fontsize=8)
for c, ch in enumerate(CHANNELS):
    ax[1].plot(meta.trial_id, amp[:, c], lw=.5, label=ch)
ax[1].set_title("Per-trial robust amplitude after cleaning (log MAD)")
ax[2].plot(meta.trial_id, jump, lw=.6, color="gray"); ax[2].axhline(CFG["session_z"], color="r", ls="--", lw=.8)
ax[2].set_title("Jump score (boundary where it exceeds the red line)"); ax[2].set_xlabel("trial_id")
rec_starts = meta.trial_id.values[np.flatnonzero(rec_break)[1:]]
for a in ax:
    for b in boundaries:
        a.axvline(meta.trial_id.values[b], color="k", alpha=.3, lw=.8)
    for r in rec_starts:
        a.axvline(r, color="r", alpha=.5, lw=1, ls="--")           # recording starts (Time gaps)
plt.tight_layout(); plt.savefig(FIG_DIR / "06_sessions.png"); plt.show()
groups = trials.loc[keep, "session_id"].to_numpy()          # recordings -> GroupKFold
norm_groups = trials.loc[keep, "regime_id"].to_numpy()       # gain regimes -> within-group normalization

## 7 · Neurophysiology checks on the cleaned data

* **Median PSD per class** (robust to the outlier sessions that dominated the means in notebook 01).
* **Band log-power** for 7 bands × 4 channels (compact replacement for the 120 redundant features).
* **Lateralization index** `LI = log10 P(C4) − log10 P(C3)`. Left-hand imagery → ERD over the **right** motor cortex
  (C4), so LI is expected to be **lower for Left than for Right**.
* Mann–Whitney U per feature with **Benjamini–Hochberg** FDR (AUC = 0.5 means no separation).

> Whole-trial band power without a baseline is a *weak* test of motor imagery (no cue timing → no ERD/ERS).
> A null result here does not prove the signal is uninformative; it simply sets the expectation for the models.

In [ ]:
def bh_fdr(p):
    p = np.asarray(p, float); n = len(p)
    order = np.argsort(p); ranks = np.empty(n); ranks[order] = np.arange(1, n + 1)
    q = p * n / ranks
    q[order] = np.minimum(np.minimum.accumulate(q[order][::-1])[::-1], 1)
    return q

# PSD of every kept trial once (Welch, 2 s windows -> 0.5 Hz resolution)
PSD = np.stack([psd(x)[1] for x in Xc])                  # (n_kept, n_freq, 4)
def band_logpower(lo, hi):
    m = (freqs >= lo) & (freqs <= hi)
    return np.log10(trapezoid(PSD[:, m, :], freqs[m], axis=1) + 1e-12)

feat = {}
for b, (lo, hi) in CFG["bands"].items():
    bp = band_logpower(lo, hi)
    for c, ch in enumerate(CHANNELS):
        feat[f"{ch}_{b}"] = bp[:, c]
for b in ("mu", "beta"):
    feat[f"LI_{b}"] = feat[f"C4_{b}"] - feat[f"C3_{b}"]
features = pd.DataFrame(feat)
features.insert(0, "trial_id", trials.loc[keep, "trial_id"].to_numpy())
features["y"] = y

rows = []
for col in [c for c in features.columns if c not in ("trial_id", "y")]:
    a, b_ = features.loc[features.y == 0, col], features.loc[features.y == 1, col]
    u, p = mannwhitneyu(a, b_)
    rows.append((col, u / (len(a) * len(b_)), p, a.median(), b_.median()))
tests = pd.DataFrame(rows, columns=["feature", "AUC_left_gt_right", "p", "median_left", "median_right"])
tests["q_fdr"] = bh_fdr(tests.p)
tests["abs_effect"] = (tests.AUC_left_gt_right - .5).abs()
display(tests.sort_values("abs_effect", ascending=False).head(10).round(4))
print("Significant after FDR (q < .05):", int((tests.q_fdr < .05).sum()), "of", len(tests))

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for c, ch in enumerate(CHANNELS):
    for cls, name in ((0, "Left"), (1, "Right")):
        axes[c].semilogy(freqs, np.median(PSD[y == cls][:, :, c], 0), label=name)
    axes[c].axvspan(*CFG["bands"]["mu"], color="g", alpha=.12); axes[c].axvspan(*CFG["bands"]["beta"], color="orange", alpha=.08)
    axes[c].set(title=f"{ch} median PSD", xlim=(1, 45), xlabel="Hz")
    mm = (freqs >= 1) & (freqs <= 45)
    v = np.median(PSD[:, mm, c], 0)
    axes[c].set_ylim(v.min() / 2, v.max() * 2)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "07_median_psd_by_class.png"); plt.show()

## 8 · Label-alignment diagnostics

The label mapping is confirmed, so the question here is: **is there decodable class information at all** with the
standard motor-imagery baseline?

1. **CSP + LDA** (8–30 Hz, no CAR) with **contiguous K-fold** (and `GroupKFold` by `session_id` when ≥ 5 sessions exist).
2. **Controls:** the same model with labels shifted by ±1, ±2, ±5 trials and with random permutations. They define what
   "chance" looks like for this exact pipeline (permutations ≈ 0.50 ± 0.02).

> CAR is **not** used here: with only 4 channels it makes the data rank-3, so CSP's covariance becomes singular.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import KFold, GroupKFold
from scipy.linalg import eigh

def csp_fit(X, y_, n_pairs):
    covs = []
    for c in (0, 1):
        A = X[y_ == c]
        C = np.einsum("nts,ntk->nsk", A, A)
        C = C / np.trace(C, axis1=1, axis2=2)[:, None, None]
        covs.append(C.mean(0) + 1e-6 * np.eye(X.shape[2]))
    vals, vecs = eigh(covs[0], covs[0] + covs[1])
    o = np.argsort(vals)
    return vecs[:, np.r_[o[:n_pairs], o[-n_pairs:]]]

def csp_feat(X, W):
    v = np.einsum("nts,sk->ntk", X, W).var(axis=1)
    return np.log(v / v.sum(1, keepdims=True))

def cv_accuracy(X, y_, groups=None):
    if groups is not None and len(np.unique(groups)) >= CFG["cv_folds"]:
        splits = GroupKFold(CFG["cv_folds"]).split(X, y_, groups)
    else:
        splits = KFold(CFG["cv_folds"], shuffle=False).split(X)
    acc = []
    for tr, te in splits:
        if len(np.unique(y_[tr])) < 2:
            continue
        W = csp_fit(X[tr], y_[tr], CFG["csp_pairs"])
        clf = LDA().fit(csp_feat(X[tr], W), y_[tr])
        acc.append(clf.score(csp_feat(X[te], W), y_[te]))
    return float(np.mean(acc)) if acc else np.nan

if len(Xc) >= 60:
    sos_m = butter(4, CFG["csp_band"], btype="bandpass", fs=CFG["fs"], output="sos")
    Xm = sosfiltfilt(sos_m, Xc, axis=1)
    y_all = meta.y.to_numpy()

    res = {"true labels": cv_accuracy(Xm, y, groups)}
    for s in (-5, -2, -1, 1, 2, 5):
        res[f"shift {s:+d}"] = cv_accuracy(Xm, np.roll(y_all, s)[keep_idx], groups)
    perm = np.array([cv_accuracy(Xm, RNG.permutation(y), groups) for _ in range(CFG["n_perm"])])
    res["random permutation (mean)"] = float(perm.mean())
    diag = pd.Series(res, name="CV accuracy").to_frame().round(3)
    display(diag)

    others = [v for k, v in res.items() if k.startswith("shift")]
    gap = res["true labels"] - max(others)
    print(f"true - best shifted = {gap:+.3f};  permutations: {perm.mean():.3f} ± {perm.std():.3f}")
    if res["true labels"] > 0.55 and gap > 0.03:
        verdict = "decodable signal FOUND (true labels clearly beat shifted / permuted labels)"
    elif res["true labels"] < 0.53:
        verdict = "NO decodable signal with whole-trial CSP + LDA (labels are confirmed, so this is a signal/feature limitation)"
    else:
        verdict = "inconclusive"
    print("Verdict:", verdict)
else:
    diag, verdict = None, "skipped (fewer than 60 kept trials)"
    print(verdict)

## 8b · Where could the signal hide?
Whole-trial band power and whole-trial CSP are at chance. Before concluding anything, three explanations are checked:

1. **Gain differences between recordings / regimes** wash out pooled features → standardize band-power features **within each gain regime**.
2. **Time dilution** – motor imagery effects are time-locked (typically a few seconds after the cue); averaging over 8 s hides them →
   **time-resolved lateralization** (1 s windows, step 0.5 s) with a **label-permutation null band**.
3. **Pooling recordings** → CSP + LDA **inside each recording**, and CSP on **2 s windows**.

> If nothing here rises above the null band / ≈ 0.55 accuracy, the honest conclusion for the report is:
> *"with these 4 channels and no cue timing, class information is not detectable with standard methods"* – which is still a valid
> finding and tells the modeling team to expect low accuracy.

In [ ]:
from scipy.stats import rankdata
fs = CFG["fs"]
if len(Xc) < 200:
    print("Skipped: fewer than 200 kept trials.")
else:
    # ---------- (1) within-session standardized band-power features
    def zscore_within(M, g):
        out = np.empty_like(M, dtype=float)
        for s_ in np.unique(g):
            m = g == s_
            out[m] = (M[m] - M[m].mean(0)) / (M[m].std(0) + 1e-9)
        return out
    cols = [c for c in features.columns if c not in ("trial_id", "y")]
    Fz = zscore_within(features[cols].to_numpy(), norm_groups)
    rows = []
    for j, c in enumerate(cols):
        a, b_ = Fz[y == 0, j], Fz[y == 1, j]
        u, p = mannwhitneyu(a, b_)
        rows.append((c, u / (len(a) * len(b_)), p))
    tests_ws = pd.DataFrame(rows, columns=["feature", "AUC_left_gt_right", "p"])
    tests_ws["q_fdr"] = bh_fdr(tests_ws.p)
    tests_ws["abs_effect"] = (tests_ws.AUC_left_gt_right - .5).abs()
    print("(1) Within-session standardized features - top 6 by |AUC - 0.5|:")
    display(tests_ws.sort_values("abs_effect", ascending=False).head(6).round(4))
    print("    significant after FDR:", int((tests_ws.q_fdr < .05).sum()), "of", len(tests_ws))

    # ---------- (2) time-resolved lateralization
    WIN, STEP = int(1.0 * fs), int(0.5 * fs)
    starts = np.arange(0, Xc.shape[1] - WIN + 1, STEP)
    t_center = (starts + WIN / 2) / fs                                             # seconds in the 10 s trial
    def win_logpower(lo, hi):
        out = np.empty((len(Xc), len(starts), 4))
        for w, s0 in enumerate(starts):
            f, P = welch(Xc[:, s0:s0 + WIN, :], fs=fs, nperseg=WIN, axis=1)       # (n, n_freq, 4)
            m = (f >= lo) & (f <= hi)
            out[:, w, :] = np.log10(P[:, m, :].mean(axis=1) + 1e-12)
        return out
    def auc_cols(M, y_):
        r = rankdata(M, axis=0); n0, n1 = (y_ == 0).sum(), (y_ == 1).sum()
        return (r[y_ == 0].sum(0) - n0 * (n0 + 1) / 2) / (n0 * n1)

    c3, c4 = CHANNELS.index("C3"), CHANNELS.index("C4")
    series = {}
    for b in ("mu", "beta"):
        lp = win_logpower(*CFG["bands"][b])
        series[f"LI_{b}"] = lp[:, :, c4] - lp[:, :, c3]
        series[f"C3_{b}"], series[f"C4_{b}"] = lp[:, :, c3], lp[:, :, c4]
    series = {k: zscore_within(v, norm_groups) for k, v in series.items()}         # within gain regime, per window
    aucs = {k: auc_cols(v, y) for k, v in series.items()}

    null = []
    for _ in range(200):
        yp = RNG.permutation(y)
        null.append(max(np.abs(auc_cols(v, yp) - .5).max() for v in series.values()))
    band = float(np.percentile(null, 95))

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
    for ax, b in zip(axes, ("mu", "beta")):
        for k in (f"LI_{b}", f"C3_{b}", f"C4_{b}"):
            ax.plot(t_center, aucs[k], marker="o", ms=3, label=k)
        ax.axhspan(.5 - band, .5 + band, color="gray", alpha=.2, label="95% permutation null (max over all curves)")
        ax.set(title=f"{b}: AUC(Left>Right) over the trial", xlabel="seconds from trial start")
    axes[0].set_ylabel("AUC"); axes[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG_DIR / "08b_time_resolved_auc.png"); plt.show()
    peak = max(((np.abs(v - .5).max(), k, float(t_center[np.abs(v - .5).argmax()])) for k, v in aucs.items()))
    print(f"(2) Largest |AUC-0.5| = {peak[0]:.3f} ({peak[1]} at ~{peak[2]:.1f} s); null band = +-{band:.3f} ->",
          "ABOVE null" if peak[0] > band else "inside null")

    # ---------- (3) CSP inside sessions, and CSP on 2 s windows
    sos_m = butter(4, CFG["csp_band"], btype="bandpass", fs=fs, output="sos")
    Xm_ = sosfiltfilt(sos_m, Xc, axis=1)
    rows = []
    for g in np.unique(groups):
        m = groups == g
        if m.sum() >= 80 and len(np.unique(y[m])) == 2:
            rows.append((f"session {int(g)}", int(m.sum()), cv_accuracy(Xm_[m], y[m])))
    sess_tbl = pd.DataFrame(rows, columns=["scope", "n_trials", "CSP+LDA acc"]).round(3)
    print("(3a) CSP + LDA inside each session (contiguous 5-fold):")
    display(sess_tbl)

    W2, S2 = int(2.0 * fs), int(1.0 * fs)
    rows = [(f"{s0 / fs:.1f}-{(s0 + W2) / fs:.1f} s", cv_accuracy(Xm_[:, s0:s0 + W2], y))
            for s0 in range(0, Xm_.shape[1] - W2 + 1, S2)]
    win_tbl = pd.DataFrame(rows, columns=["window (raw trial time)", "CSP+LDA acc"]).round(3)
    print("(3b) CSP + LDA on 2 s windows (pooled, contiguous 5-fold):")
    display(win_tbl)
    best = max(sess_tbl["CSP+LDA acc"].max() if len(sess_tbl) else 0, win_tbl["CSP+LDA acc"].max())
    print(f"Best accuracy found in 8b: {best:.3f}  (chance = 0.50 +- ~0.02 for the pooled data)")

## 9 · Hand-off package for the modeling team
Everything the next person needs, with a **data contract** written to `manifest.json`.

| File | Content |
|---|---|
| `X_clean.npy` | `(n, 2500, 4)` float32 (10 s per trial), filtered as a continuous stream, scaled per trial, **no re-reference** → use for CSP / classical ML |
| `X_clean_car.npy` | same with common-average reference → optional for deep models |
| `y.npy` | `0 = Left`, `1 = Right` |
| `trials_kept.csv` | one row per row of `X_clean` (`trial_id`, `file_name`, `label`, `session_id` (recording), `block_id`, `regime_id`, `scale`, …) |
| `trials_all.csv` | every trial with `kept` / `reject_reason` (the documented rejection list) |
| `features_clean.csv` | 30 compact band-power features + `y` (alternative to the 120 raw features) |
| `config.json`, `manifest.json` | parameters, shapes, class balance, caveats |

In [ ]:
def one_scale(A):
    s = np.sqrt(np.median((A ** 2).mean(axis=2), axis=1))      # median power over time: ignores bursts
    return (A / s[:, None, None]).astype(np.float32), s

Xs, scale = one_scale(Xc)
Xcar_s, _ = one_scale(Xc - Xc.mean(axis=2, keepdims=True))

kept_tbl = trials.loc[keep, ["trial_id", "file_name", "label", "y", "session_id",
                             "recording_id", "block_id", "regime_id",
                             "stale_frac", "max_stale_run", "contam_frac", "max_abs_z"]].reset_index(drop=True)
kept_tbl["scale"] = scale

np.save(OUT_DIR / "X_clean.npy", Xs)
np.save(OUT_DIR / "X_clean_car.npy", Xcar_s)
np.save(OUT_DIR / "y.npy", y.astype(np.int8))
kept_tbl.to_csv(OUT_DIR / "trials_kept.csv", index=False)
trials.drop(columns=["clean_std_mean"]).to_csv(OUT_DIR / "trials_all.csv", index=False)
features.to_csv(OUT_DIR / "features_clean.csv", index=False)
tests.to_csv(OUT_DIR / "band_power_tests.csv", index=False)

cfg_out = {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}
cfg_out["bands"] = {k: list(v) for k, v in CFG["bands"].items()}
(OUT_DIR / "config.json").write_text(json.dumps(cfg_out, indent=2))

manifest = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "n_trials_loaded": int(len(trials)), "n_trials_kept": int(keep.sum()),
    "X_clean": {"shape": list(Xs.shape), "dtype": "float32", "axes": ["trial", "time", "channel"], "channels": CHANNELS,
                "fs_hz": CFG["fs"], "duration_s": Xs.shape[1] / CFG["fs"]},
    "labels": {"encoding": {"Left": 0, "Right": 1}, "counts": {"Left": int((y == 0).sum()), "Right": int((y == 1).sum())},
               "mapping": "label[i] <-> cellula_MI_data_{i}.csv (first 2160 of 2400 labels); confirmed by the project owner"},
    "preprocessing": ["stale rows interpolated", "DC removal", f"notch {CFG['notch_hz']} Hz (Q={CFG['notch_q']})",
                      f"band-pass {CFG['band']} Hz, Butterworth order {CFG['band_order']}, zero-phase",
                      "each continuous block (consecutive files) filtered as ONE stream; first/last trial of every block rejected", "one robust (median-power) scalar per trial; inter-channel ratios preserved"],
    "rejection": {"stale_frac_gt": CFG["max_stale_frac"], "stale_run_gt": CFG["max_stale_run"],
                  "contam_frac_gt": CFG["max_contam_frac"], "abs_z_gt": CFG["max_abs_z"],
                  "by_reason": trials.loc[~trials.kept, "reject_reason"].str.split("+").explode().value_counts().to_dict()},
    "cv_advice": "session_id = recording (from Time gaps). Use GroupKFold on session_id, or contiguous (non-shuffled) KFold. Never shuffle trials across recordings.",
    "n_recordings": int(trials.session_id.nunique()), "n_filter_blocks": int(trials.block_id.nunique()),
    "caveats": ["no cue timing -> no ERD/ERS or baseline correction",
                "CAR on 4 channels is rank-3: do not use X_clean_car for CSP"],
    "decodability_check": verdict,
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

# ---- contract test: reload what was written and assert the promises
Xl, yl, tl = np.load(OUT_DIR / "X_clean.npy"), np.load(OUT_DIR / "y.npy"), pd.read_csv(OUT_DIR / "trials_kept.csv")
assert Xl.shape[0] == len(yl) == len(tl) and Xl.shape[2] == 4
assert np.isfinite(Xl).all() and set(np.unique(yl)) <= {0, 1}
assert (tl.y.to_numpy() == yl).all()
print("Hand-off package written and verified ->", OUT_DIR)
print(json.dumps({k: manifest[k] for k in ("n_trials_loaded", "n_trials_kept")}, indent=2))

### Starter snippet for the modeling team (not executed here)
```python
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, KFold

X = np.load("outputs_clean/X_clean.npy")            # (n, 2500, 4)  use this for CSP + SVM
y = np.load("outputs_clean/y.npy")
meta = pd.read_csv("outputs_clean/trials_kept.csv")

# leakage-safe splitting: whole recordings stay together (fall back to contiguous KFold if <5 sessions)
splitter = GroupKFold(5).split(X, y, meta.session_id) if meta.session_id.nunique() >= 5 else KFold(5, shuffle=False).split(X)
# session_id = recording (exact, from the Time column); fit CSP / scalers on X[tr] only
for tr, te in splitter:
    ...   # fit CSP / scaler on X[tr] ONLY
```

## 10 · Summary, decision log & open questions

In [ ]:
summary_tbl = pd.Series({
    "Trials loaded": len(trials),
    "Trials kept": int(keep.sum()),
    "Rejected": int((~keep).sum()),
    "Kept %": f"{keep.mean():.1%}",
    "Class balance (Left/Right)": f"{(y == 0).sum()} / {(y == 1).sum()}",
    "Recordings / filter blocks / gain regimes": f"{trials.session_id.nunique()} / {trials.block_id.nunique()} / {trials.regime_id.nunique()}",
    "Sampling rate (median, from Time)": f"{np.median(eff_rate):.1f} Hz",
    "Peaks at 50 Hz (evidence)": f"{share_50:.0%}",
    "Band-power features significant (FDR)": f"{int((tests.q_fdr < .05).sum())} / {len(tests)}",
    "Decodability check (section 8)": verdict,
}, name="value").to_frame()
display(summary_tbl)

### Decision log
| # | Decision | Why |
|---|---|---|
| D1 | `trial_id` = file number (1-based) is the only key | removes the 0-/1-based shift between notebook-01 tables |
| D2 | fs = 250 Hz | median duration rate ≈ 250 Hz **and** mains peak lands at 50 Hz in ~all trials |
| D3 | Index = time, no timestamp resampling | `Time` is read-time; signal is periodic in index space |
| D11 | Files are consecutive 10 s slices → filter each continuous block as one stream; reject first/last trial of every block | per-file filtering created 1-2 s edge transients and forced an 8 s trim |
| D12 | `session_id` = recording from `Time` gaps; `regime_id` = gain heuristic | exact groups for CV; regimes only for normalization |
| D4 | Stale rows interpolated; trials with long runs or > 15 % contaminated area (±0.2 s around runs) rejected | they are held values, not EEG, and interpolation makes the filters ring |
| D5 | Notch 50 Hz + band-pass 1–40 Hz, zero-phase | 50 Hz carries > 90 % of raw power; modeling narrows to 8–30 Hz |
| D6 | One robust (median-power) scalar per trial (not per-channel z-score) | preserves inter-channel variance ratios used by CSP |
| D7 | Two arrays: with / without CAR | 4-channel CAR makes the data rank-3 (breaks CSP) |
| D8 | Contiguous / session-grouped CV only | sessions differ strongly in gain; random folds leak |
| D10 | Labels treated as confirmed (owner) | section 8 reports decodability, not alignment |
| D9 | 120 raw features replaced by 30 compact band-power features | the 120 were ~80 % redundant and computed on unfiltered data |

### Open questions (cannot be settled from the data)
1. **Unmatched labels** – what are the 240 labels (2161…2400) without files? (The mapping of the first 2160 is confirmed.)
2. **Cue timing / baseline** – needed for ERD/ERS and baseline correction. Is the whole 10 s trial the imagery period? Section 8b looks for the time window where the classes differ.
3. **Units** – raw values have a DC level of ~300 000; the µV scale is unknown (cleaned std is ≈ 10–20 in good sessions).
4. **Subjects** – is it one subject or several? Sessions in section 6 may be subjects.
5. **Reference scheme / hardware** – would explain the stale rows and the very large 50 Hz line.

### What to retire from notebook 01
Keep it as the **audit trail**. For the final report you can drop the repeated redundancy / reliability audits (cells ~84–88,
103–126, 143–147, 165–211): the conclusions are in the decision log above.